In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from pathlib import Path
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

FEATURES_DIR = Path('../results/features')
MODEL_DIR    = Path('../results/models')
FIG_DIR      = Path('../results/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

# cargar modelo SVM y scaler
svm      = joblib.load(MODEL_DIR / 'svm_clasico.pkl')
scaler   = joblib.load(MODEL_DIR / 'scaler_clasico.pkl')
features = joblib.load(MODEL_DIR / 'features_clasico.pkl')

# cargar features de test
df_test  = pd.read_csv(FEATURES_DIR / 'features_test.csv')
X_test   = df_test[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y_test   = df_test['clase'].values
Xte      = scaler.transform(X_test)
clases   = sorted(np.unique(y_test))

print(f'Test: {Xte.shape}')
print(f'Clases: {clases}')

In [ ]:
THETA = 0.60  # umbral de confianza

proba        = svm.predict_proba(Xte)
confianza    = proba.max(axis=1)
pred_svm     = svm.predict(Xte)

# separar imágenes por confianza
mask_alta    = confianza >= THETA
mask_baja    = confianza < THETA

print(f'Imágenes con confianza >= {THETA}: {mask_alta.sum()} ({mask_alta.mean():.1%})')
print(f'Imágenes con confianza <  {THETA}: {mask_baja.sum()} ({mask_baja.mean():.1%})')
print(f'\nAccuracy SVM solo: {accuracy_score(y_test, pred_svm):.3f}')

# distribución de confianza
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(confianza, bins=50, color='steelblue', edgecolor='white')
ax.axvline(THETA, color='red', linestyle='--', label=f'θ = {THETA}')
ax.set_title('Distribución de confianza del SVM')
ax.set_xlabel('Confianza (probabilidad máxima)')
ax.set_ylabel('Frecuencia')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / '06_distribucion_confianza_svm.png', dpi=150)
plt.show()

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

PROCESSED_DIR = Path('../data/processed')
IMG_SIZE      = (224, 224)
BATCH_SIZE    = 32
EPOCHS        = 10
N_CLASES      = len(clases)

# generadores
datagen_train = ImageDataGenerator(rescale=1./255)
datagen_val   = ImageDataGenerator(rescale=1./255)

gen_train = datagen_train.flow_from_directory(
    PROCESSED_DIR / 'train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)
gen_val = datagen_val.flow_from_directory(
    PROCESSED_DIR / 'val',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# modelo
base    = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base.trainable = False

x       = base.output
x       = GlobalAveragePooling2D()(x)
x       = Dropout(0.3)(x)
output  = Dense(N_CLASES, activation='softmax')(x)

modelo_cnn = Model(inputs=base.input, outputs=output)
modelo_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = modelo_cnn.fit(
    gen_train,
    validation_data=gen_val,
    epochs=EPOCHS
)

modelo_cnn.save(MODEL_DIR / 'cnn_verificador.h5')
print('CNN guardada.')

In [ ]:
def reconstruir_ruta(nombre_archivo, clase):
    ruta = PROCESSED_DIR / 'test' / clase / nombre_archivo
    if ruta.exists():
        return ruta
    return None

rutas_test = [reconstruir_ruta(row['archivo'], row['clase']) 
              for _, row in df_test.iterrows()]

# verificar cuántas rutas encontró
encontradas = sum(1 for r in rutas_test if r is not None)
print(f'Rutas encontradas: {encontradas}/{len(rutas_test)}')

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image as keras_image

cnn = load_model(MODEL_DIR / 'cnn_verificador.h5')

idx_to_clase = {v: k for k, v in gen_val.class_indices.items()}

def predecir_cnn(img_path):
    img  = keras_image.load_img(img_path, target_size=IMG_SIZE)
    arr  = keras_image.img_to_array(img) / 255.0
    arr  = np.expand_dims(arr, axis=0)
    prob = cnn.predict(arr, verbose=0)
    return idx_to_clase[np.argmax(prob)]

pred_hibrido = []
fuente       = []

for i, (ruta, conf, pred_s) in enumerate(zip(rutas_test, confianza, pred_svm)):
    if conf >= THETA:
        pred_hibrido.append(pred_s)
        fuente.append('SVM')
    else:
        pred_hibrido.append(predecir_cnn(ruta))
        fuente.append('CNN')

pred_hibrido = np.array(pred_hibrido)

print(f'Resueltos por SVM : {fuente.count("SVM")} ({fuente.count("SVM")/len(fuente):.1%})')
print(f'Resueltos por CNN : {fuente.count("CNN")} ({fuente.count("CNN")/len(fuente):.1%})')
print(f'\nAccuracy SVM solo  : {accuracy_score(y_test, pred_svm):.3f}')
print(f'Accuracy híbrido   : {accuracy_score(y_test, pred_hibrido):.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, (pred, titulo) in zip(axes, [
    (pred_svm,     f'SVM solo (acc={accuracy_score(y_test, pred_svm):.1%})'),
    (pred_hibrido, f'Híbrido (acc={accuracy_score(y_test, pred_hibrido):.1%})')
]):
    cm = confusion_matrix(y_test, pred, labels=clases)
    ConfusionMatrixDisplay(cm, display_labels=clases).plot(ax=ax, cmap='Greens', colorbar=False, xticks_rotation=45)
    ax.set_title(titulo)

plt.suptitle('SVM solo vs Pipeline híbrido SVM + CNN', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / '06_comparacion_svm_vs_hibrido.png', dpi=150)
plt.show()

In [ ]:
import joblib

joblib.dump(pred_svm,      MODEL_DIR / 'pred_svm.pkl')
joblib.dump(pred_hibrido,  MODEL_DIR / 'pred_hibrido.pkl')
joblib.dump(y_test,        MODEL_DIR / 'y_test.pkl')
joblib.dump(fuente,        MODEL_DIR / 'fuente.pkl')
joblib.dump(confianza,     MODEL_DIR / 'confianza.pkl')
joblib.dump(history.history, MODEL_DIR / 'cnn_history.pkl')


print('Variables guardadas.')